# Size a slate with KellyIQ

Send your bankroll and the bets you have chosen, with your view of each, and get back a stake for
every bet, sized across the whole slate together.

Running the optimize cell uses **one optimize** from your daily allowance; every other call here is
free. KellyIQ sizes the bets you choose: it does not choose bets or predict outcomes, and its
results are calculations, not betting advice. 21+, where legal.

In [ ]:
%pip install --quiet kellyiq pandas

## Connect

Create an API key in the KellyIQ app under **Settings, API** (every plan includes it, Free too).
Set `KELLYIQ_API_KEY` before starting Jupyter, or paste the key when asked. It is not stored in the
notebook.

In [ ]:
import os
from getpass import getpass

import pandas as pd
from kellyiq import KellyIQ

client = KellyIQ(api_key=os.environ.get("KELLYIQ_API_KEY") or getpass("KellyIQ API key: "))

me = client.me()
print(f"Plan: {me['tier']}, optimizes left today: {me['limits']['optimize']['remainingToday']}")

In [ ]:
# The sports you can size today, and the key to send for each. Free.
pd.DataFrame(client.sports())

## Your slate

One row per bet, with **both sides' prices** (KellyIQ removes the bookmaker's margin using the
other side). `confidence` is your view of each bet: `STRONG_EDGE`, `MODERATE_EDGE`, `SMALL_EDGE` or
`MARKET` (no edge). On Bettor and up you can send your own win probability instead: replace the
`confidence` column with a `probability` column (for example `0.58`).

Replace these rows with your own.

In [ ]:
slate = pd.DataFrame([
    {"sport": "baseball_mlb", "team": "Dodgers", "opponent": "Padres",    "odds": -145, "opponentOdds": 125,  "confidence": "MODERATE_EDGE"},
    {"sport": "baseball_mlb", "team": "Cubs",    "opponent": "Cardinals", "odds": 120,  "opponentOdds": -140, "confidence": "MODERATE_EDGE"},
    {"sport": "baseball_mlb", "team": "Mets",    "opponent": "Braves",    "odds": 135,  "opponentOdds": -155, "confidence": "MARKET"},
])
bankroll = 1000
slate

## Optimize

This cell uses one optimize. Errors arrive as specific exceptions, so you can handle each case. If
it prints an error, fix the slate and run it again before the cells below.

In [ ]:
from kellyiq import DailyLimitError, SlateNotOptimizableError, UpgradeRequiredError, ValidationError

picks = slate.assign(id=slate.index.astype(str)).to_dict("records")
try:
    result = client.optimize({"bankroll": bankroll, "picks": picks})
except ValidationError as e:
    print("Fix these and run again:", *e.violations, sep="\n  ")
except UpgradeRequiredError as e:
    print(f"Needs the {e.required_tier} plan:", *e.violations, sep="\n  ")
except SlateNotOptimizableError as e:
    print("KellyIQ could not size this slate:", *e.violations, sep="\n  ")
except DailyLimitError as e:
    print(f"No optimizes left today. The allowance resets at {e.resets_at}.")
else:
    alloc = result["allocation"]
    share = f"{alloc['kellyMultiplier']:.0%} of full Kelly" if alloc["kellyMultiplier"] is not None else "no stake"
    print(f"Total stake ${alloc['totalStake']:.2f} of ${bankroll}, {share} (seed {result['seed']})")

## Stakes, and why

`market` is the fair probability with the margin removed; `yours` is that probability moved by your
view, and is what drove the stake. `expected_value` is expected profit per $1 staked. A bet with no
stake has a `reason` (`NO_EDGE` means your view puts no edge on that price).

In [ ]:
stakes = {s["id"]: s["stake"] for s in result["allocation"]["stakes"]}
pd.DataFrame([
    {
        "team": p["team"],
        "odds": p["odds"],
        "stake": stakes.get(p["id"], 0.0),
        "market": (p["probability"] or {}).get("market"),
        "yours": (p["probability"] or {}).get("yours"),
        "expected_value": p["expectedValue"],
        "reason": p["reason"],
    }
    for p in result["picks"]
])

## Risk, and other Kelly fractions

A simulation of the slate (bets treated as independent) for the allocation above and for full, half
and quarter Kelly. `prob_loss` is the chance the slate loses money overall; `var5` is the result
that 1 slate in 20 ends at or below. Smaller fractions mean smaller swings and slower growth; which
to use is your call.

In [ ]:
modes = ["riskAdjusted", "full", "half", "quarter"]
pd.DataFrame([
    {
        "sizing": m,
        "total_stake": round(sum((p["sizing"] or {}).get(m, {}).get("stake", 0) for p in result["picks"]
                                 if (p["sizing"] or {}).get(m)), 2),
        "mean_profit": sim["meanProfit"],
        "prob_loss": sim["probLoss"],
        "var5": sim["var5"],
    }
    for m in modes
    for sim in [(result["risk"].get(m) or {}).get("independent") or {}]
]).set_index("sizing")

## Same answer again

Send the same input with the returned `seed` to reproduce this result exactly on the same engine
version (`result["engineVersion"]`). Like any optimize, a re-run uses one from your allowance.

```python
again = client.optimize({"bankroll": bankroll, "picks": picks, "seed": result["seed"]})
```

More: [API reference](https://www.kellyiq.com/developers/reference/optimize) ·
[Python SDK](https://www.kellyiq.com/developers/python) ·
[all examples](https://github.com/kellyiq/kellyiq-examples)